# Hard-wall S100 domain-wall correlator scaling

This independent lane runs 72 fixed 25-trajectory tasks at $N_x=20$ and
$N_y=24,28,32$. It stages the tested bundle under `/content`, streams nested
task/cycle progress, and resumes from one rolling 16-cycle checkpoint or
verified completed-task pairs.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('[notebook] Google Drive mounted', flush=True)


In [ ]:
from pathlib import Path
import json
import torch

CONSTRUCTION = 'hard'
REPORT_ONLY = False
MAX_NEW_TASKS = None  # Set to 1 for a timing/restart test.
BUNDLE_DRIVE_DIR = Path('/content/drive/MyDrive/final_production_new_designs/08_domain_wall_correlator_scaling')
OUTPUT_ROOT = Path('/content/drive/MyDrive/classA_final_production_outputs/domain_wall_correlator_nx20_ny24-32_a1-1-3_nsh1-2-dense_s100_2ny_raster_v1')
LOCAL_BUNDLE_DIR = Path('/content/08_domain_wall_correlator_scaling_hard')
SCRATCH_ROOT = Path('/content/domain_wall_correlator_scratch_hard')
CONFIG = {
    'sampling_revision': 'domain_wall_correlator_nx20_ny24-32_a1-1-3_nsh1-2-dense_s100_2ny_raster_v1',
    'root_seed': 2026090408,
    'Nx': 20,
    'Ny_values': [24, 28, 32],
    'alpha_1_values': [1.0, 3.0],
    'alpha_2': 30.0,
    'nshell_values': [1, 2, None],
    'samples_per_case': 100,
    'batch_size': 25,
    'cycles_multiplier': 2,
    'segment_cycles': 16,
    'trial_orbitals': 'X',
    'filling_fraction': 0.5,
    'n_a': 0.5,
    'init_mode': 'default',
    'sequence': 'raster_y',
    'perfect_correction': True,
    'postselect': False,
    'postselect_probability': 0.0,
    'dtype': 'complex128',
    'device': 'cuda:0',
    'backend_by_nshell': {'1': 'local', '2': 'local', 'dense': 'dense'},
    'state_representation': 'physical_frame',
    'triv_region_local_mode': False,
    'frame_reorthonormalize_interval': 1,
    'constructions': {
        'hard': {'DW': True, 'dw_truncation': True, 'meas_slab_only': True},
        'soft': {'DW': True, 'dw_truncation': False, 'meas_slab_only': False},
    },
    'observables': {
        'cycles': '0..2Ny inclusive',
        'x_resolved_square_correlator': True,
        'legacy_xavg_square_correlator': True,
        'global_charge': True,
    },
}
if not torch.cuda.is_available():
    raise RuntimeError('This production notebook requires CUDA.')
gpu_name = torch.cuda.get_device_name(0)
gpu_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
if 'A100' not in gpu_name.upper() or gpu_gib < 35:
    raise RuntimeError(f'Expected an A100 40-GB-class GPU, got {gpu_name} ({gpu_gib:.2f} GiB)')
if torch.complex128 != getattr(torch, CONFIG['dtype']):
    raise RuntimeError('Locked dtype is not complex128.')
print(json.dumps({
    'construction': CONSTRUCTION,
    'config': CONFIG,
    'workload': {'configurations': 18, 'tasks': 72, 'trajectories': 1800},
    'bundle': str(BUNDLE_DRIVE_DIR),
    'canonical_gpu_source': str(BUNDLE_DRIVE_DIR / 'src/classA_U1FGTN_gpu.py'),
    'occupied_frame_source': str(BUNDLE_DRIVE_DIR / 'src/occupied_frame_gpu.py'),
    'output': str(OUTPUT_ROOT),
    'scratch': str(SCRATCH_ROOT),
    'gpu': gpu_name,
    'gpu_GiB': gpu_gib,
    'dtype': CONFIG['dtype'],
    'report_only': REPORT_ONLY,
    'max_new_tasks': MAX_NEW_TASKS,
}, indent=2, sort_keys=True))


In [ ]:
import codecs
import os
import shutil
import subprocess
import sys

def _stream_process_output(process):
    decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    while True:
        chunk = os.read(process.stdout.fileno(), 8192)
        if not chunk:
            break
        text = decoder.decode(chunk)
        if text:
            sys.stdout.write(text)
            sys.stdout.flush()
    tail = decoder.decode(b'', final=True)
    if tail:
        sys.stdout.write(tail)
        sys.stdout.flush()
    return process.wait()

if not BUNDLE_DRIVE_DIR.is_dir():
    raise FileNotFoundError(f'Missing Drive bundle: {BUNDLE_DRIVE_DIR}')
if LOCAL_BUNDLE_DIR.exists():
    shutil.rmtree(LOCAL_BUNDLE_DIR)
shutil.copytree(BUNDLE_DRIVE_DIR, LOCAL_BUNDLE_DIR)
config_path = Path('/content/domain_wall_correlator_config_hard.json')
config_path.write_text(json.dumps(CONFIG, indent=2, sort_keys=True) + '\n', encoding='utf-8')
command = [
    sys.executable, '-u', str(LOCAL_BUNDLE_DIR / 'run_campaign.py'),
    '--config', str(config_path),
    '--construction', CONSTRUCTION,
    '--output-root', str(OUTPUT_ROOT),
    '--scratch-root', str(SCRATCH_ROOT),
]
if REPORT_ONLY:
    command.append('--report-only')
if MAX_NEW_TASKS is not None:
    command.extend(['--max-new-tasks', str(int(MAX_NEW_TASKS))])
print('[launch] ' + ' '.join(command), flush=True)
process = subprocess.Popen(
    command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0
)
try:
    returncode = _stream_process_output(process)
finally:
    if process.poll() is None:
        process.terminate()
        process.wait(timeout=30)
if returncode:
    raise subprocess.CalledProcessError(returncode, command)
print('[notebook] hard-wall runner exited successfully', flush=True)


In [ ]:
from google.colab import runtime
runtime.unassign()
print('done')
